# Hierarchical Architecture — Google ADK

**Pattern:** Root agent delegates to sub-agents  
**Framework:** Google ADK  

```
User Query
     │
     ▼
┌──────────────────┐
│  travel_manager  │  ← root agent (no tools, only sub_agents)
└────────┬─────────┘
         │ delegates via transfer
    ┌────┴────┐
    ▼         ▼
┌────────┐ ┌──────────┐
│weather │ │ safety   │
│specialist│ │specialist│
└────────┘ └──────────┘
```

**ADK `sub_agents=`:** The root agent can transfer control to specialist sub-agents.  
Sub-agents complete their work and return results to the root for synthesis.

In [ ]:
import asyncio
import os
from dotenv import load_dotenv
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
print("✓ Ready")

In [ ]:
def get_weather(city: str) -> dict:
    """Get weather and score for a city.
    Args:
        city: City name.
    Returns:
        Dict with condition, temperature, and score.
    """
    data = {
        "tokyo": {"condition": "Clear", "temp_c": 18, "score": 9},
        "paris": {"condition": "Partly Cloudy", "temp_c": 16, "score": 7},
        "bangalore": {"condition": "Rainy", "temp_c": 25, "score": 6},
    }
    key = city.lower()
    return {"city": city, **data[key]} if key in data else {"error": f"No data for '{city}'."}

def get_safety(city: str) -> dict:
    """Get safety advisory and score for a city.
    Args:
        city: City name.
    Returns:
        Dict with level, notes, and score.
    """
    data = {
        "tokyo": {"level": "Low", "notes": "Very safe.", "score": 10},
        "paris": {"level": "Low", "notes": "Alert in crowded spots.", "score": 8},
        "bangalore": {"level": "Medium", "notes": "Monsoon affects transport.", "score": 6},
    }
    key = city.lower()
    return {"city": city, **data[key]} if key in data else {"error": f"No data for '{city}'."}

print("Tools ready")

In [ ]:
# Sub-agents (specialists)
weather_agent = Agent(
    name="weather_specialist",
    model="gemini-2.0-flash",
    description="Researches weather conditions for cities. Use for any weather queries.",
    instruction="Call get_weather() for each requested city. Return structured weather data.",
    tools=[get_weather],
)

safety_agent = Agent(
    name="safety_specialist",
    model="gemini-2.0-flash",
    description="Researches travel safety advisories. Use for any safety queries.",
    instruction="Call get_safety() for each requested city. Return structured safety data.",
    tools=[get_safety],
)

# Root manager agent — no tools, only sub_agents
manager_agent = Agent(
    name="travel_manager",
    model="gemini-2.0-flash",
    description="Travel report manager that coordinates research and produces final reports.",
    instruction="""You are a travel project manager. When asked to compare cities:

1. Delegate weather research to weather_specialist for all cities.
2. Delegate safety research to safety_specialist for all cities.
3. Synthesize into a report:
   - '## Travel Report' header
   - '### [City]' section with weather + safety per city
   - '## Top Pick' with the best city and reason""",
    sub_agents=[weather_agent, safety_agent],
)

print(f"Manager: {manager_agent.name}")
print(f"Sub-agents: {[a.name for a in manager_agent.sub_agents]}")

In [ ]:
async def run_hierarchical(cities: list) -> str:
    session_service = InMemorySessionService()
    session = await session_service.create_session(
        app_name="travel_manager", user_id="user_01"
    )
    runner = InMemoryRunner(agent=manager_agent, session_service=session_service)
    query = f"Compare for travel: {', '.join(cities)}. Produce a full hierarchical report."

    print(f"Query: {query}\n")
    final_response = ""
    async for event in runner.run_async(
        user_id=session.user_id, session_id=session.id,
        new_message=genai_types.Content(role="user", parts=[genai_types.Part(text=query)]),
    ):
        if hasattr(event, "tool_call") and event.tool_call:
            print(f"  [tool] {event.tool_call.name}({str(event.tool_call.args)[:50]})")  
        elif event.is_final_response() and event.content:
            for part in event.content.parts:
                if part.text: final_response += part.text
    return final_response

report = await run_hierarchical(["Tokyo", "Paris"])
print("\n--- Final Report ---")
print(report)

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| `sub_agents=[...]` on root agent | ADK's native multi-agent hierarchy — no custom routing code |
| Root agent has no tools | Manager only orchestrates — tools live on specialists |
| `description=` on sub-agents | Manager uses descriptions to decide which agent to call |

### Hierarchical: All 4 Frameworks
| Framework | Manager Mechanism | Worker Assignment |
|---|---|---|
| LangChain | Python function calls | Explicit code flow |
| LangGraph | Supervisor node + routing | Conditional edges |
| CrewAI | `Process.hierarchical` + `manager_llm` | Auto-delegated by manager LLM |
| ADK | `sub_agents=` + transfer | Manager reads sub-agent descriptions |

### Next: [04-Orchestrator-Subagent →](../../04-Orchestrator-Subagent/LangChain/orchestrator.ipynb)